# RLOO with increasing response length

**The adaptive-length runs outperform the historical fixed-length policy and preserve boxed answers. Both target alphas beat low-temperature sampling at pass@1, but neither beats it clearly at answer pass@4. Effective alpha stays near 1; raising the target to 4 does not resolve this. All methods, including MCMC, are compared on the same 500 questions below.**

## Shared protocol

Qwen2.5-0.5B on the same 1,024 GSM8K training questions, seed 0, physical GPU 1. Independent runs start from identical base/LoRA initialization; the settings differ only in target alpha (2 or 4). The cap follows **64 → 128 → 192 → 256 → 320 → 384 → 448 → 512**, with **38 updates and 2,432 rollouts per stage** (304 updates, 19,456 responses per run). Training including diagnostics took **58.3 / 58.0 minutes** for alpha 2 / 4. The quota was calibrated once and reused.

All-linear LoRA: rank 8, scaling alpha 16, dropout 0. AdamW: lr 1e-5, weight decay 0, gradient clipping 1, no warmup. Each update uses 16 questions × 4 responses, generation batch 64, scoring microbatch 4, BF16, T=1 and unrestricted token sampling. The reward is $r=\alpha\log p_0-\log\pi$, with a leave-one-out baseline and one update per fresh rollout batch. Answer labels are used only for evaluation. EOS contributes to sequence scores; unfinished responses are censored at the cap.

```text
Can you solve the following math problem? {question} Please reason step by step, and put your final answer within \boxed{{}}.
```

The literal double boxed braces match the historical GSM8K experiment. [Alpha-2 settings](settings.json) · [Alpha-4 settings](settings_alpha4.json).

## Training trajectories

Each point uses the same **16 held-out questions × 4 fresh current-policy responses**, T=1 and a fixed 512-token cap. Time includes diagnostics; curves connect recorded points without smoothing. Probe accuracy estimates pass@1 on this small diagnostic set, separately from the final test evaluation.

![Policy NLL, reference CE and combined true loss](figures/loss_decomposition.png)

Policy NLL is $E_\pi[-\log\pi]$, which estimates policy entropy; reference CE is $E_\pi[-\log p_0]$. The third subplot combines them as $L=\alpha\,\mathrm{CE}-\mathrm{NLL}=E_\pi[\log\pi-\alpha\log p_0]$. All three use the same on-policy samples. Values are absolute nats per response, without length normalization; CE is shown before multiplication by alpha. Error bars are pointwise 1.96 × question SE. True loss is reverse KL up to a constant at fixed alpha and horizon. Different alphas define different targets, so their absolute true losses are not a common KL scale. [Decomposed values and source hashes](results/loss_decomposition.json).

![Effective alpha, accuracy and response length](figures/alpha_comparison.png)

Effective-alpha bars show question-slope SD, not confidence intervals or intrinsic temperature noise.

Target alpha 2: true loss **250.8 → 53.1**; final effective alpha **0.989**, question SD **0.250** (fixed-response estimate 1.010). Target alpha 4: true loss **752.5 → 142.8**; final effective alpha **1.022**, question SD **0.754** (fixed-response estimate 1.011). Both losses fall, but neither diagnostic supports learning the target power exponent.

The similar NLL/CE shapes are not identical values: $\mathrm{CE}=\mathrm{NLL}+D_{KL}(\pi\Vert p_0)$. Both start at the same value because the policy initially equals the base. Their shared early decrease dominates the plot scale; the final CE minus NLL estimates are **13.37 / 16.14 nats** for target alpha 2 / 4. The first two panels use identical axis limits. In the true loss, policy NLL enters with a minus sign. Independent rescoring of all 128 final probe responses with full logits and a separately loaded base agrees with the saved scores to within 0.040 nat per response. No duplicated-array or adapter-switch error was found in these checks. [Scoring audit](results/loss_audit.json).

## Final evaluation: 500 questions

Same test questions, 32 independent responses per question and a 512-token cap. Policies use T=1. Answer grading accepts explicit final answers; boxed grading also requires the requested format. Low T=0.25 was selected on the historical development set.

| Method | Answer pass@1 | Answer pass@4 | Answer pass@32 | Boxed pass@1 | Median tokens |
|---|---:|---:|---:|---:|---:|
| Base T=1 | 16.71% | 41.48% | 76.40% | 14.23% | 303 |
| Fixed-length RLOO, alpha 2 | 27.08% | 47.17% | 73.80% | 0.62% | 88 |
| Base T=0.25 | 34.02% | 58.98% | 82.40% | 31.64% | 271 |
| Adaptive RLOO, alpha 2 | 38.56% | 59.25% | 85.80% | 38.55% | 256 |
| Adaptive RLOO, alpha 4 | 38.72% | 58.38% | 81.40% | 38.71% | 257 |
| MCMC, alpha 2 | 38.93% | 65.82% | 86.80% | 35.27% | 237 |

Against low T, answer pass@1 gains are +4.54 pp (95% CI [+2.72, +6.34]) for alpha 2 and +4.69 pp (95% CI [+2.77, +6.61]) for alpha 4. Both answer pass@4 difference intervals cross zero. Alpha 4 versus alpha 2 has no clear pass@1 or pass@4 advantage; its pass@32 difference is -4.40 pp (95% CI [-7.20, -1.60]).

These are 10,000-replicate paired-question bootstrap intervals. One training seed does not measure training variability. The historical fixed-length run had fewer updates and different diagnostic overhead/scoring kernels; its comparison does not isolate the causal effect of the curriculum. [All metrics and intervals](results/summary.json).

## MCMC interpretation and cost

The GSM8K baseline MCMC evaluation is complete on the same **500 questions × 32 chains** used in the table above. MCMC targets alpha 2 with proposal T=0.5, block size 32 and two MH updates per block. Model revision, prompt, engine settings, seed rule and 512-token cap match the original evaluation. It uses the local full-suffix acceptance ratio and retains EOS; it is a finite-compute baseline, not a verified exact power sampler or exact upstream reproduction. No matching-alpha-4 MCMC result is available here.

Answer pass@4, policy minus MCMC: -6.57 pp (95% CI [-8.42, -4.74]) for alpha 2 and -7.44 pp (95% CI [-9.49, -5.49]) for alpha 4. Answer pass@1 differences are -0.37 pp (95% CI [-2.04, +1.24]) / -0.21 pp (95% CI [-1.98, +1.52]) respectively.

MCMC generated **26,318,722 tokens**, including rejected proposals, versus **4,271,691 / 4,283,649** for the alpha-2 / alpha-4 policies (6.16× / 6.14×). These counts exclude reference scoring and training; compute is not matched. [Unified metrics, source hashes and intervals](results/summary.json) · [Original MCMC completion](../gsm8k_power_policy/results/evaluation/test/mcmc_alpha2/completion.json).